In [ ]:
       
import sys, time, os, urllib.parse
import numpy as np
import torch
import torch.nn.functional as F
import requests
from bs4 import BeautifulSoup
from bs4.element import Comment
from transformers import AutoTokenizer, AutoModelForCausalLM
PORT = 3000
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"   
ADD_SPECIAL_TOKENS = True
EPISODES = 20                     
MAX_STEPS_PER_EPISODE = 15
TRUST_REMOTE_CODE = False  
# 3) Globals & model load
WEBSHOP_URL = f"http://localhost:{PORT}"
COACH_PERIOD = 4                  

use_bf16 = torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
dtype = torch.bfloat16 if use_bf16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, padding_side="left", use_fast=True, trust_remote_code=TRUST_REMOTE_CODE
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=dtype, device_map="auto", trust_remote_code=TRUST_REMOTE_CODE
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model.eval()

def llm(prompt: str, stop=["\n"]) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,  # greedy
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
            return_dict_in_generate=True,
        )
    gen_ids = out.sequences[:, inputs.input_ids.shape[1]:]
    text = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0]

    if stop:
        cut = len(text)
        for s in stop:
            pos = text.find(s)
            if pos != -1:
                cut = min(cut, pos)
        text = text[:cut]
    return text.strip()

# 5) WebShop helpers
ACTION_TO_TEMPLATE = {
    'Description': 'description_page.html',
    'Features': 'features_page.html',
    'Reviews': 'review_page.html',
    'Attributes': 'attributes_page.html',
}

def clean_str(p):
    try:
        return p.encode().decode("unicode-escape").encode("latin1").decode("utf-8")
    except Exception:
        return str(p)

def tag_visible(element):
    ignore = {'style', 'script', 'head', 'title', 'meta', '[document]'}
    return (element.parent and element.parent.name not in ignore and not isinstance(element, Comment))

def webshop_text(session, page_type, query_string='', page_num=1, asin='', options=None, subpage='', **kwargs):
    if options is None:
        options = {}
    if page_type == 'init':
        url = f'{WEBSHOP_URL}/{session}'
    elif page_type == 'search':
        url = f'{WEBSHOP_URL}/search_results/{session}/{query_string}/{page_num}'
    elif page_type == 'item':
        url = f'{WEBSHOP_URL}/item_page/{session}/{asin}/{query_string}/{page_num}/{options}'
    elif page_type == 'item_sub':
        url = f'{WEBSHOP_URL}/item_sub_page/{session}/{asin}/{query_string}/{page_num}/{subpage}/{options}'
    elif page_type == 'end':
        safe_options = urllib.parse.quote(str(options))
        url = f'{WEBSHOP_URL}/done/{session}/{asin}/{safe_options}'
    else:
        url = f'{WEBSHOP_URL}/{session}'

    html = requests.get(url).text
    html_obj = BeautifulSoup(html, 'html.parser')
    texts = html_obj.find_all(string=True)  
    visible_texts = list(filter(tag_visible, texts))

    observation = ''
    option_type = ''
    options_map = {}
    asins = []
    cnt = 0
    prod_cnt = 0
    just_prod = 0
    for t in visible_texts:
        if t == '\n': 
            continue
        if t.replace('\n', '').replace('\\n', '').replace(' ', '') == '':
            continue
        if t.parent.name == 'button':
            processed_t = f'\n[{t}] '
        elif t.parent.name == 'label':
            if f"'{t}'" in url:
                processed_t = f'[[{t}]]'
            else:
                processed_t = f'[{t}]'
            options_map[str(t)] = option_type
        elif t.parent.get('class') == ["product-link"]:
            processed_t = f'\n[{t}] '
            if prod_cnt >= 3:
                processed_t = ''
            prod_cnt += 1
            asins.append(str(t))
            just_prod = 0
        else:
            processed_t = '\n' + str(t) + ' '
            if cnt < 2 and page_type != 'init':
                processed_t = ''
            if just_prod <= 2 and prod_cnt >= 4:
                processed_t = ''
            option_type = str(t)
            cnt += 1
        just_prod += 1
        observation += processed_t

    info = {}
    if options_map:
        info['option_types'] = options_map
    if asins:
        info['asins'] = asins
    if 'Your score (min 0.0, max 1.0)' in visible_texts:
        idx = visible_texts.index('Your score (min 0.0, max 1.0)')
        info['reward'] = float(visible_texts[idx + 1])
        observation = 'Your score (min 0.0, max 1.0): ' + (visible_texts[idx + 1])
    return clean_str(observation), info

class webshopEnv:
    def __init__(self):
        self.sessions = {}
    def step(self, session, action):
        done = False
        observation_ = None
        if action == 'reset':
            self.sessions[session] = {'session': session, 'page_type': 'init'}
        elif action.startswith('think['):
            observation = 'OK.'
        elif action.startswith('search['):
            assert self.sessions[session]['page_type'] == 'init'
            query = action[7:-1]
            self.sessions[session] = {
                'session': session, 'page_type': 'search',
                'query_string': query, 'page_num': 1
            }
        elif action.startswith('click['):
            button = action[6:-1]
            if button == 'Buy Now':
                assert self.sessions[session]['page_type'] == 'item'
                self.sessions[session]['page_type'] = 'end'
                done = True
            elif button == 'Back to Search':
                assert self.sessions[session]['page_type'] in ['search', 'item_sub', 'item']
                self.sessions[session] = {'session': session, 'page_type': 'init'}
            elif button == 'Next >':
                assert False  
            elif button == '< Prev':
                assert self.sessions[session]['page_type'] in ['search', 'item_sub', 'item']
                if self.sessions[session]['page_type'] == 'search':
                    assert False
                elif self.sessions[session]['page_type'] == 'item_sub':
                    self.sessions[session]['page_type'] = 'item'
                elif self.sessions[session]['page_type'] == 'item':
                    self.sessions[session]['page_type'] = 'search'
                    self.sessions[session]['options'] = {}
            elif button in ACTION_TO_TEMPLATE:
                assert self.sessions[session]['page_type'] == 'item'
                self.sessions[session]['page_type'] = 'item_sub'
                self.sessions[session]['subpage'] = button
            else:
                if self.sessions[session]['page_type'] == 'search':
                    assert button in self.sessions[session].get('asins', [])
                    self.sessions[session]['page_type'] = 'item'
                    self.sessions[session]['asin'] = button
                elif self.sessions[session]['page_type'] == 'item':
                    assert 'option_types' in self.sessions[session]
                    assert button in self.sessions[session]['option_types'], (button, self.sessions[session]['option_types'])
                    option_type = self.sessions[session]['option_types'][button]
                    if 'options' not in self.sessions[session]:
                        self.sessions[session]['options'] = {}
                    self.sessions[session]['options'][option_type] = button
                    observation_ = f'You have clicked {button}.'
        else:
            assert False

        observation, info = webshop_text(**self.sessions[session])
        if observation_:
            observation = observation_
        self.sessions[session].update(info)
        reward = info.get('reward', 0.0)
        return observation, reward, done

env = webshopEnv()

# 6) Profiles & seeds
COACH_PROFILE = """
[COACH_PROFILE]
You are an expert web shopping coach. Your job is to observe the player's actions and provide concise, actionable advice when needed.

If the player is stuck or repeating actions: Tell them to Refine your search. or Check product options before buying.

If the player has found the right item but hasn't bought it: Remind them to If all conditions are met, click [Buy Now.]

If the player is performing well: Offer encouragement with a phrase like Continue as you are.

Keep your advice brief and to the point.
"""

PLAYER_PROFILE = """
[PLAYER_PROFILE]
You are a web shopping assistant. Your mission is to complete the user's request by following these steps:

Search: Use the search[query] action to find the product.
Inspect: Use the click[product_id] action to view product details.
Validate: Check the product to ensure it meets all user criteria (e.g., price, size, color).
Purchase: Use the click[Buy Now] action to complete the task if all criteria are met.
Always output one action at a time. Your actions must be in the correct format.
"""

COACH_prompt = """
Action: reset
Observation: 
Webshop 
Instruction:  
i am looking for slim jeans that are granite color and machine washable, and price lower than 70.00 dollars 
[Search] 

Action: search[slim granite jeans machine washable]
Observation: 
[Back to Search] 
Page 1 (Total results: 50) 
[Next >] 
[B085RFQVMJ] 
HUNGSON Men's Stretchy Ripped Skinny Jeans Taped Slim Fit Denim Jeans 
$19.99 to $33.99 

Action: click[B085RFQVMJ]
Observation: 
[Back to Search] 
[< Prev] 
color [black-1][black2021][more ripped black][black][more ripped blue][white][white tape print][blue]
size [28][30][32][34][36][38][40]
HUNGSON Men's Stretchy Ripped Skinny Jeans Taped Slim Fit Denim Jeans 
Price: $19.99 to $33.99 
Rating: N.A. 
[Description] 
[Features] 
[Reviews] 
[Buy Now] 

[COACH_ADVICE]
You chose the wrong option.
No "granite" color here. Refine search for "granite jeans".
[/COACH_ADVICE]
"""

COACH_promptact = """
Action: reset
Observation: 
Webshop 
Instruction:  
i need some fully cooked canned meats with a long shelf life, and price lower than 120.00 dollars 
[Search] 

Action: search[fully cooked canned meats long shelf life]
Observation: 
[Back to Search] 
Page 1 (Total results: 50) 
[Next >] 
[B000FIZVYW] 
HORMEL Premium Turkey Breast in Water, 5 Ounce (Pack of 12) 
$18.41 

Action: click[B000FIZVYW]
Observation: 
[Back to Search] 
[< Prev] 
HORMEL Premium Turkey Breast in Water, 5 Ounce (Pack of 12) 
Price: $18.41 
Rating: N.A. 
[Description] 
[Features] 
[Reviews] 
[Buy Now] 

[COACH_ADVICE]
Looks like a match. If requirements are met, click [Buy Now].
[/COACH_ADVICE]
"""

COACH_SEED_MODE = "full"   # "full" or "actonly"
COACH_SEED = COACH_prompt if COACH_SEED_MODE == "full" else COACH_promptact

def build_coach_input(history_text: str, coach_seed: str) -> str:
    return f"""{COACH_PROFILE}

### COACH SEED
{coach_seed.strip()}

History:
{history_text}

Give advice for the next steps.
"""

def call_coach(history_text: str) -> str:
    raw = llm(
        build_coach_input(history_text, coach_seed=COACH_SEED),
        stop=["\n\nAction:", "\n\n[", "\n\n", "Stop after the checklist.", "Stop after"]
    )
    return raw.strip()

# Player prompts
prompt1 = """Webshop 
Instruction:  
i would like a 3 ounce bottle of bright citrus deodorant for sensitive skin, and price lower than 50.00 dollars 
[Search]  

Action: search[3 ounce bright citrus deodorant sensitive skin]
Observation: 
[Back to Search] 
Page 1 (Total results: 50) 
[Next >] 
[B078GWRC1J] 
Bright Citrus Deodorant by Earth Mama | Natural and Safe for Sensitive Skin, Pregnancy and Breastfeeding, Contains Organic Calendula 3-Ounce 
$10.99 
[B078GTKVXY] 
Ginger Fresh Deodorant by Earth Mama | Natural and Safe for Sensitive Skin, Pregnancy and Breastfeeding, Contains Organic Calendula 3-Ounce 
$10.99 
[B08KBVJ4XN] 
Barrel and Oak - Aluminum-Free Deodorant, Deodorant for Men, Essential Oil-Based Scent, 24-Hour Odor Protection, Cedar & Patchouli Blend, Gentle on Sensitive Skin (Mountain Sage, 2.7 oz, 2-Pack) 
$15.95  

Action: think[B078GWRC1J and B078GTKVXY are bright citrus deodorant less then 50 dollars. I can check B078GWRC1J first.]
Observation: OK.

Action: click[B078GWRC1J]
Observation: 
[Back to Search] 
[< Prev] 
scent [assorted scents][bright citrus][calming lavender][ginger fresh][simply non-scents]
size [travel set (4-pack)][3 ounce (pack of 1)][3-ounce (2-pack)]
Bright Citrus Deodorant by Earth Mama | Natural and Safe for Sensitive Skin, Pregnancy and Breastfeeding, Contains Organic Calendula 3-Ounce 
Price: $10.99 
Rating: N.A. 
[Description] 
[Features] 
[Reviews] 
[Buy Now]  

[COACH_ADVICE]
You have selected the right product. Now, verify the options (scent and size) to make sure they match the instruction.
[/COACH_ADVICE]

Action: think[For 3 ounce bottle of bright citrus deodorant for sensitive skin, the item has options 'bright citrus' and '3 ounce (pack of 1)' and seems good to buy.]
Observation: OK.

Action: click[bright citrus]
Observation: You have clicked bright citrus. 

Action: click[3 ounce (pack of 1)]
Observation: You have clicked 3 ounce (pack of 1). 

Action: click[Buy Now]
"""

prompt1_actonly = """Webshop 
Instruction:  
i would like a 3 ounce bottle of bright citrus deodorant for sensitive skin, and price lower than 50.00 dollars 
[Search]  

Action: search[3 ounce bright citrus deodorant sensitive skin]
Observation: 
[Back to Search] 
Page 1 (Total results: 50) 
[Next >] 
[B078GWRC1J] 
Bright Citrus Deodorant by Earth Mama | Natural and Safe for Sensitive Skin, Pregnancy and Breastfeeding, Contains Organic Calendula 3-Ounce 
$10.99 
[B078GTKVXY] 
Ginger Fresh Deodorant by Earth Mama | Natural and Safe for Sensitive Skin, Pregnancy and Breastfeeding, Contains Organic Calendula 3-Ounce 
$10.99 
[B08KBVJ4XN] 
Barrel and Oak - Aluminum-Free Deodorant, Deodorant for Men, Essential Oil-Based Scent, 24-Hour Odor Protection, Cedar & Patchouli Blend, Gentle on Sensitive Skin (Mountain Sage, 2.7 oz, 2-Pack) 
$15.95  

Action: click[B078GWRC1J]
Observation: 
[Back to Search] 
[< Prev] 
scent [assorted scents][bright citrus][calming lavender][ginger fresh][simply non-scents]
size [travel set (4-pack)][3 ounce (pack of 1)][3-ounce (2-pack)]
Bright Citrus Deodorant by Earth Mama | Natural and Safe for Sensitive Skin, Pregnancy and Breastfeeding, Contains Organic Calendula 3-Ounce 
Price: $10.99 
Rating: N.A. 
[Description] 
[Features] 
[Reviews] 
[Buy Now]  

[COACH_ADVICE]
Check the product options to ensure it meets all criteria.
[/COACH_ADVICE]

Action: click[bright citrus]
Observation: You have clicked bright citrus. 

Action: click[3 ounce (pack of 1)]
Observation: You have clicked 3 ounce (pack of 1). 

Action: click[Buy Now]
"""

def webshop_run(idx, prompt, to_print=True, coach_period=COACH_PERIOD, max_steps=MAX_STEPS_PER_EPISODE):
    action = 'reset'
    init_prompt = PLAYER_PROFILE.strip() + "\n\n" + prompt
    history = ''
    obs_count = 0

    for i in range(max_steps):
        try:
            observation, reward, done = env.step(idx, action)
        except AssertionError:
            observation, reward, done = 'Invalid action!', 0, False

        if action.startswith('think'):
            observation = 'OK.'

        obs_count += 1

        if to_print:
            print(f'Action: {action}\nObservation: {observation}\n')
            sys.stdout.flush()

        if i:
            history += f' {action}\nObservation: {observation}\n\nAction:'
        else:
            history += f'{observation}\n\nAction:'

        if coach_period and (obs_count % coach_period == 0):
            hist_for_coach = (init_prompt + history[:-len("Action:")]) if history.endswith("Action:") else (init_prompt + history)
            advice = call_coach(hist_for_coach)
            if history.endswith("Action:"):
                history = history[:-len("Action:")] + f"[COACH_ADVICE]\n{advice}\n[/COACH_ADVICE]\n\nAction:"
            else:
                history += f"\n[COACH_ADVICE]\n{advice}\n[/COACH_ADVICE]\n\nAction:"
            if to_print:
                print(f"[COACH_ADVICE]\n{advice}\n[/COACH_ADVICE]\n")
                sys.stdout.flush()

        if done:
            return reward

        action = llm(init_prompt + history, stop=['\n']).lstrip(' ')

    return 0

def run_episodes(prompt, n=EPISODES):
    rs, cnt = [], 0
    start_all = time.time()
    for i in range(n):
        print('-----------------')
        print(i)
        start_ep = time.time()
        try:
            r = webshop_run(f'fixed_{i}', prompt, to_print=True)
        except (AssertionError, torch.cuda.OutOfMemoryError) as e:
            print(f"Error in episode {i}: {e}")
            r = 0
            cnt += 1
        end_ep = time.time()
        print(f"Episode {i} took {end_ep - start_ep:.2f} seconds")
        rs.append(r)
        r_avg = sum(rs) / len(rs)
        sr = len([_ for _ in rs if _ == 1]) / len(rs)
        fr = cnt / len(rs)
        print(i+1, r_avg, sr, fr)
        print('-------------')
    total = time.time() - start_all
    print(f"Total {n} episodes took {total:.2f} seconds")
    print(f"Average per episode: {total/n:.2f} seconds")
    r_final = sum(rs) / len(rs)
    sr_final = len([_ for _ in rs if _ == 1]) / n
    fr_final = cnt / n
    print(r_final, sr_final, fr_final)
    return rs

res1 = run_episodes(prompt1, n=EPISODES)
